# TuranAssist – классификатор интентов на multilingual-e5-small

Ноутбук обучает трансформерную модель для чат-бота TuranAssist и сравнивает её с baseline (TF-IDF).

1. Клонирует репозиторий с GitHub: наборы фраз и общий код оценки `model/common` – тот же, что у baseline.
2. **Вариант A – замороженные эмбеддинги** `intfloat/multilingual-e5-small` + логистическая регрессия.
3. **Вариант B – дообучение** e5-small целиком с классификационной головой.
4. Выбирает вариант по точности 5-кратной кросс-валидации (как baseline), подбирает порог «не понял».
5. Экспортирует выбранную модель в ONNX (с softmax на выходе), квантует в int8, сверяет точность и задержку.
6. Загружает модель, токенизатор и отчёты в репозиторий Hugging Face Hub.

**Перед запуском:**
- Среда выполнения → Сменить среду → **T4 GPU**.
- Слева «Ключи» (Secrets) → добавьте секрет `HF_TOKEN` – токен Hugging Face с правом **write**
  (huggingface.co/settings/tokens) и включите ему доступ к ноутбуку. Токен в ноутбуке не печатается.
- В первой ячейке впишите свой `HF_REPO`.

Полный прогон на T4 занимает около 15–20 минут.

In [1]:
# настройки запуска
GITHUB_REPO = "https://github.com/AlexTrav/TuranAssist.git"
HF_REPO = "AlexCode2003/turanassist-intent-e5"   # <аккаунт Hugging Face>/<имя репозитория модели>
BASE_MODEL = "intfloat/multilingual-e5-small"
SEED = 42
MAX_LEN = 64          # вопросы короткие: 64 токенов хватает даже длинным формулировкам
EPOCHS = 10
LEARNING_RATE = 5e-5
BATCH_SIZE = 32

In [2]:
# зависимости и код проекта: наборы фраз и общий модуль оценки берутся прямо из репозитория
!pip -q install transformers onnx onnxruntime huggingface_hub scikit-learn pyyaml psutil
!rm -rf /content/TuranAssist && git clone -q --depth 1 {GITHUB_REPO} /content/TuranAssist

import os, sys
os.environ["DATA_DIR"] = "/content/TuranAssist/data"
sys.path.insert(0, "/content/TuranAssist")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 75.5 MB/s eta 0:00:00


In [3]:
import json, random, time
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from model.common import data, metrics

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

ood_val, ood_test = data.load_ood()
sets = {"train": data.load_train(), "test": data.load_test(), "ood_val": ood_val, "ood_test": ood_test,
        "external": data.load_external(), "scenarios": data.load_scenarios()}
X = [e.text for e in sets["train"]]
y = [e.labels[0] for e in sets["train"]]
INTENTS = sorted(set(y))
label2id = {l: i for i, l in enumerate(INTENTS)}
y_ids = np.array([label2id[l] for l in y])
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=data.SEED).split(X, y))
print({k: len(v) for k, v in sets.items()}, "интентов:", len(INTENTS))

устройство: cuda Tesla T4
{'train': 1166, 'test': 318, 'ood_val': 77, 'ood_test': 78, 'external': 209, 'scenarios': 65} интентов: 53


## Общие функции

e5 требует префикс `query: ` перед текстом; эмбеддинг – среднее по токенам (mean pooling) с L2-нормировкой.

In [4]:
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def with_prefix(texts):
    return ["query: " + t for t in texts]

def encode(texts):
    return tokenizer(with_prefix(texts), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")

def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).float()
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def embed(encoder, texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        batch = {k: v.to(device) for k, v in encode(texts[i:i + bs]).items()}
        hidden = encoder(**batch).last_hidden_state
        out.append(torch.nn.functional.normalize(mean_pool(hidden, batch["attention_mask"]), dim=-1).cpu())
    return torch.cat(out).numpy()

# оценка варианта на всех наборах – так же, как у baseline (model/common/metrics.py)
def evaluate_variant(proba_fn, oof_proba, threshold=None):
    oof_conf = oof_proba.max(1)
    oof_correct = [INTENTS[i] == t for i, t in zip(oof_proba.argmax(1), y)]
    result = {"cv_accuracy": float(np.mean(oof_correct))}
    if threshold is None:
        ood_conf = proba_fn([e.text for e in sets["ood_val"]]).max(1)
        threshold, info = metrics.choose_threshold(oof_correct, oof_conf, ood_conf)
        result["threshold_selection"] = info
    result["threshold"] = float(threshold)
    for key in ("test", "ood_test", "external", "scenarios"):
        proba = proba_fn([e.text for e in sets[key]])
        pred, conf = [INTENTS[i] for i in proba.argmax(1)], proba.max(1)
        result[key] = metrics.evaluate_set(sets[key], pred, conf, threshold)
        if key == "test":
            true = [e.labels[0] for e in sets[key]]
            f1, per_intent = metrics.macro_f1(true, pred, INTENTS)
            result[key].update({"macro_f1_no_threshold": f1, "per_intent": per_intent,
                                "confusions": metrics.top_confusions(true, pred)})
    return result

def summary(name, r):
    t = r["test"]
    print(f"{name}: cv={r['cv_accuracy']:.3f} test={t['in_domain_top1_no_threshold']['value']:.3f} "
          f"f1={t['macro_f1_no_threshold']:.3f} thr={r['threshold']:.2f} "
          f"ood={r['ood_test']['ood_rejected']['value']:.3f} ext={r['external']['overall']['value']:.3f} "
          f"scen={r['scenarios']['overall']['value']:.3f}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

## Вариант A – замороженные эмбеддинги + логистическая регрессия

In [5]:
encoder = AutoModel.from_pretrained(BASE_MODEL).to(device).eval()
E_train = embed(encoder, X)

# C подбираем по кросс-валидации, как у baseline
best_c, best_acc, oof_frozen = None, -1.0, None
for c in (1, 10, 100):
    oof = np.zeros((len(X), len(INTENTS)))
    for tr, va in folds:
        clf = LogisticRegression(C=c, max_iter=5000).fit(E_train[tr], y_ids[tr])
        oof[va] = clf.predict_proba(E_train[va])
    acc = float(np.mean(oof.argmax(1) == y_ids))
    print(f"C={c}: cv accuracy {acc:.3f}")
    if acc > best_acc:
        best_c, best_acc, oof_frozen = c, acc, oof

head = LogisticRegression(C=best_c, max_iter=5000).fit(E_train, y_ids)
frozen_proba = lambda texts: head.predict_proba(embed(encoder, texts))
results = {"e5_frozen_logreg": evaluate_variant(frozen_proba, oof_frozen)}
results["e5_frozen_logreg"]["C"] = best_c
summary("e5_frozen_logreg", results["e5_frozen_logreg"])

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C=1: cv accuracy 0.669
C=10: cv accuracy 0.761
C=100: cv accuracy 0.801
e5_frozen_logreg: cv=0.801 test=0.893 f1=0.896 thr=0.59 ood=0.859 ext=0.598 scen=0.800


## Вариант B – дообучение e5-small с классификационной головой

5 фолдов кросс-валидации (для честной оценки и подбора порога) и финальное обучение на всех фразах.

In [6]:
def finetune(texts, labels, seed=SEED):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=len(INTENTS), id2label=dict(enumerate(INTENTS)), label2id=label2id).to(device)
    enc = encode(list(texts))
    dataset = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))
    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                                         generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    total = EPOCHS * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
    model.train()
    for epoch in range(EPOCHS):
        for ids, mask, lab in loader:
            loss = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=lab.to(device)).loss
            loss.backward()
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
    return model.eval()

@torch.no_grad()
def finetuned_proba(model, texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        batch = {k: v.to(device) for k, v in encode(texts[i:i + bs]).items()}
        out.append(torch.softmax(model(**batch).logits, dim=-1).cpu())
    return torch.cat(out).numpy()

start = time.time()
oof_ft = np.zeros((len(X), len(INTENTS)))
X_arr = np.array(X, dtype=object)
for k, (tr, va) in enumerate(folds, 1):
    fold_model = finetune(X_arr[tr], y_ids[tr])
    oof_ft[va] = finetuned_proba(fold_model, list(X_arr[va]))
    print(f"фолд {k}: accuracy {np.mean(oof_ft[va].argmax(1) == y_ids[va]):.3f}")
    del fold_model; torch.cuda.empty_cache()

ft_model = finetune(X_arr, y_ids)
train_minutes = (time.time() - start) / 60
results["e5_finetuned"] = evaluate_variant(lambda t: finetuned_proba(ft_model, t), oof_ft)
results["e5_finetuned"]["train_minutes_cv_and_final"] = train_minutes
summary("e5_finetuned", results["e5_finetuned"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-small
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


фолд 1: accuracy 0.650


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-small
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] BertForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-small
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


фолд 5: accuracy 0.652


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-small
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


e5_finetuned: cv=0.624 test=0.695 f1=0.660 thr=0.05 ood=0.987 ext=0.378 scen=0.262


## Выбор варианта и экспорт в ONNX

Вариант выбирается по точности кросс-валидации (не по тесту). Экспортируемая модель сразу выдаёт
вероятности интентов: для варианта A в граф входят энкодер, mean pooling, нормировка и линейная голова
логистической регрессии, для варианта B – дообученная модель и softmax. Бэкенду не важно, какой вариант победил.

In [7]:
best = max(results, key=lambda n: results[n]["cv_accuracy"])
print("выбран:", best)

class FrozenWithHead(torch.nn.Module):
    def __init__(self, encoder, coef, intercept):
        super().__init__()
        self.encoder = encoder
        self.linear = torch.nn.Linear(coef.shape[1], coef.shape[0])
        self.linear.weight.data = torch.tensor(coef, dtype=torch.float32)
        self.linear.bias.data = torch.tensor(intercept, dtype=torch.float32)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        emb = torch.nn.functional.normalize(mean_pool(hidden, attention_mask), dim=-1)
        return torch.softmax(self.linear(emb), dim=-1)

class FinetunedWithSoftmax(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        return torch.softmax(self.model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=-1)

# экспортируем копию на CPU: исходные модели остаются на GPU для дальнейшей сверки
import copy
export_module = (FrozenWithHead(copy.deepcopy(encoder), head.coef_, head.intercept_) if best == "e5_frozen_logreg"
                 else FinetunedWithSoftmax(copy.deepcopy(ft_model))).cpu().eval()

EXPORT_DIR = "/content/export"
os.makedirs(EXPORT_DIR, exist_ok=True)
fp32_path, int8_path = f"{EXPORT_DIR}/model_fp32.onnx", f"{EXPORT_DIR}/model.onnx"
sample = encode(["пример вопроса"])
export_args = dict(input_names=["input_ids", "attention_mask"], output_names=["probabilities"],
                   dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                                 "probabilities": {0: "batch"}}, opset_version=17)
inputs = (sample["input_ids"], sample["attention_mask"])
try:  # классический экспортёр TorchScript – самый предсказуемый для BERT-подобных моделей
    torch.onnx.export(export_module, inputs, fp32_path, dynamo=False, **export_args)
except TypeError:  # старые версии torch не знают параметра dynamo
    torch.onnx.export(export_module, inputs, fp32_path, **export_args)
except Exception as exc:  # если в новой версии torch классический экспортёр убран – новый dynamo-экспортёр
    print("TorchScript-экспорт недоступен:", exc)
    torch.onnx.export(export_module, inputs, fp32_path, dynamo=True, **export_args)

# динамическая int8-квантизация весов: модель в ~4 раза меньше и быстрее на CPU бесплатного хостинга
from onnxruntime.quantization import QuantType, quantize_dynamic
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8)
tokenizer.save_pretrained(EXPORT_DIR)
for p in (fp32_path, int8_path):
    print(os.path.basename(p), f"{os.path.getsize(p) / 2**20:.1f} МБ")

выбран: e5_frozen_logreg


/tmp/ipykernel_649/3698701847.py:39: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(export_module, inputs, fp32_path, dynamo=False, **export_args)
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:214: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/transformers/integrations/sdpa_attention.

model_fp32.onnx 448.5 МБ
model.onnx 112.7 МБ


## Проверка ONNX: точность после квантизации, задержка и память на CPU

Задержка меряется на CPU в один поток – так ближе к бесплатному хостингу (0,1 CPU на Render
будет медленнее; живые замеры – на странице «Производительность» и в Задании 2).

In [8]:
import onnxruntime as ort, psutil

def ort_session(path):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1
    opts.inter_op_num_threads = 1
    return ort.InferenceSession(path, opts, providers=["CPUExecutionProvider"])

def ort_proba(session, texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(with_prefix(texts[i:i + bs]), padding=True, truncation=True, max_length=MAX_LEN,
                        return_tensors="np")
        out.append(session.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                      "attention_mask": enc["attention_mask"].astype(np.int64)})[0])
    return np.concatenate(out)

process = psutil.Process()
onnx_report = {}
test_texts = [e.text for e in sets["test"]]
torch_test = (frozen_proba if best == "e5_frozen_logreg" else (lambda t: finetuned_proba(ft_model, t)))(test_texts)
for name, path in (("fp32", fp32_path), ("int8", int8_path)):
    rss_before = process.memory_info().rss
    session = ort_session(path)
    rss_after = process.memory_info().rss
    proba = ort_proba(session, test_texts)
    onnx_report[name] = {
        "size_mb": os.path.getsize(path) / 2**20,
        "session_rss_mb": (rss_after - rss_before) / 2**20,
        "max_abs_diff_vs_torch": float(np.abs(proba - torch_test).max()),
        "same_top1_as_torch": float(np.mean(proba.argmax(1) == torch_test.argmax(1))),
        "latency_ms": metrics.latency_ms(lambda t: ort_proba(session, [t]), test_texts),
    }
    print(name, json.dumps({k: v for k, v in onnx_report[name].items() if k != "latency_ms"}),
          "p50/p95:", round(onnx_report[name]["latency_ms"]["p50"], 1), round(onnx_report[name]["latency_ms"]["p95"], 1))

# итоговые метрики – у квантованной модели, которая и пойдёт в продакшн; порог – от исходной модели
int8_session = ort_session(int8_path)
results[best + "_onnx_int8"] = evaluate_variant(lambda t: ort_proba(int8_session, t),
                                                (oof_frozen if best == "e5_frozen_logreg" else oof_ft),
                                                threshold=results[best]["threshold"])
results[best + "_onnx_int8"]["latency_ms"] = onnx_report["int8"]["latency_ms"]
summary(best + "_onnx_int8", results[best + "_onnx_int8"])

fp32 {"size_mb": 448.54836654663086, "session_rss_mb": 375.23828125, "max_abs_diff_vs_torch": 1.3191812801016667e-06, "same_top1_as_torch": 1.0} p50/p95: 12.1 21.1
int8 {"size_mb": 112.65746593475342, "session_rss_mb": -302.66796875, "max_abs_diff_vs_torch": 0.4540705633354699, "same_top1_as_torch": 0.949685534591195} p50/p95: 6.6 13.7
e5_frozen_logreg_onnx_int8: cv=0.801 test=0.881 f1=0.884 thr=0.59 ood=0.859 ext=0.579 scen=0.723


## Отчёт и сравнение с baseline

In [9]:
baseline = json.load(open("/content/TuranAssist/model/reports/baseline/metrics.json", encoding="utf-8"))
b_name = baseline["best"]
rows = {f"baseline: {b_name}": baseline["results"][b_name], **results}

def fmt(p):
    return f"{p['value']:.3f} [{p['ci95'][0]:.3f}; {p['ci95'][1]:.3f}]"

lines = ["# Трансформер multilingual-e5-small против baseline", "",
         "Значения – доля верных ответов, в скобках 95% доверительный интервал Уилсона.", "",
         "| Модель | CV acc | Test top-1 | Test macro-F1 | Порог | Test с порогом | OOD отклонено | Внешний тест | Сценарии | p50, мс | p95, мс |",
         "|---|---|---|---|---|---|---|---|---|---|---|"]
for name, r in rows.items():
    lat = r.get("latency_ms", {})
    p50 = f"{lat['p50']:.1f}" if lat else "–"
    p95 = f"{lat['p95']:.1f}" if lat else "–"
    lines.append(f"| `{name}` | {r['cv_accuracy']:.3f} | {fmt(r['test']['in_domain_top1_no_threshold'])} "
                 f"| {r['test']['macro_f1_no_threshold']:.3f} | {r['threshold']:.3f} | {fmt(r['test']['overall'])} "
                 f"| {fmt(r['ood_test']['ood_rejected'])} | {fmt(r['external']['overall'])} "
                 f"| {fmt(r['scenarios']['overall'])} | {p50} | {p95} |")
final = results[best + "_onnx_int8"]
lines += ["", f"Выбран вариант `{best}` (по точности кросс-валидации), в продакшн идёт ONNX int8.", "",
          "ONNX: размер, память сессии, совпадение с PyTorch:", ""]
for name, o in onnx_report.items():
    lines.append(f"- {name}: {o['size_mb']:.0f} МБ, память сессии ~{o['session_rss_mb']:.0f} МБ, "
                 f"совпадение top-1 с PyTorch {o['same_top1_as_torch']:.3f}, "
                 f"p50 {o['latency_ms']['p50']:.1f} мс, p95 {o['latency_ms']['p95']:.1f} мс (CPU, 1 поток)")
lines += ["", "Точность по стилям теста (int8, с порогом):", ""]
lines += [f"- {g}: {fmt(p)}" for g, p in final["test"]["by_group"].items()]
lines += ["", "По языкам (int8, с порогом):", ""]
lines += [f"- {g}: {fmt(p)}" for g, p in final["test"]["by_lang"].items()]
lines += ["", "Самые слабые интенты (F1):", ""]
weakest = sorted(final["test"]["per_intent"].items(), key=lambda kv: kv[1]["f1"])[:8]
lines += [f"- `{k}`: F1 {v['f1']:.2f}" for k, v in weakest]
lines += ["", "Частые ошибки (верный → предсказанный):", ""]
lines += [f"- `{c['true']}` → `{c['pred']}`: {c['count']}" for c in final["test"]["confusions"]]
report_md = "\n".join(lines) + "\n"
print(report_md)

model_info = {
    "name": best, "base_model": BASE_MODEL, "format": "onnx-int8", "file": "model.onnx",
    "inputs": ["input_ids", "attention_mask"], "output": "probabilities", "query_prefix": "query: ",
    "max_length": MAX_LEN, "threshold": results[best]["threshold"], "intents": INTENTS,
    "hf_repo": HF_REPO, "seed": SEED, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
}
json.dump(model_info, open(f"{EXPORT_DIR}/model_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump({"best": best, "results": results, "onnx": onnx_report}, open(f"{EXPORT_DIR}/metrics.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=float)
open(f"{EXPORT_DIR}/report.md", "w", encoding="utf-8").write(report_md)
open(f"{EXPORT_DIR}/README.md", "w", encoding="utf-8").write(
    "---\nlicense: mit\nbase_model: intfloat/multilingual-e5-small\nlanguage: [ru, kk, en]\n"
    "pipeline_tag: text-classification\n---\n\n# TuranAssist intent classifier\n\n"
    "Классификатор 53 интентов чат-бота TuranAssist (Университет «Туран»), ONNX int8.\n"
    "Код и данные: https://github.com/AlexTrav/TuranAssist\n\n" + report_md)
os.remove(fp32_path)  # в репозиторий модели кладём только int8 – её использует бэкенд
print(sorted(os.listdir(EXPORT_DIR)))

# Трансформер multilingual-e5-small против baseline

Значения – доля верных ответов, в скобках 95% доверительный интервал Уилсона.

| Модель | CV acc | Test top-1 | Test macro-F1 | Порог | Test с порогом | OOD отклонено | Внешний тест | Сценарии | p50, мс | p95, мс |
|---|---|---|---|---|---|---|---|---|---|---|
| `baseline: tfidf_char` | 0.598 | 0.805 [0.758; 0.845] | 0.811 | 0.356 | 0.698 [0.646; 0.746] | 0.808 [0.707; 0.880] | 0.636 [0.569; 0.699] | 0.831 [0.722; 0.903] | 0.8 | 1.6 |
| `e5_frozen_logreg` | 0.801 | 0.893 [0.854; 0.922] | 0.896 | 0.589 | 0.629 [0.575; 0.680] | 0.859 [0.765; 0.919] | 0.598 [0.530; 0.662] | 0.800 [0.687; 0.879] | – | – |
| `e5_finetuned` | 0.624 | 0.695 [0.642; 0.743] | 0.660 | 0.049 | 0.129 [0.096; 0.170] | 0.987 [0.931; 0.998] | 0.378 [0.315; 0.445] | 0.262 [0.170; 0.380] | – | – |
| `e5_frozen_logreg_onnx_int8` | 0.801 | 0.881 [0.840; 0.912] | 0.884 | 0.589 | 0.566 [0.511; 0.619] | 0.859 [0.765; 0.919] | 0.579 [0.511; 0.644] | 0.723 [0.604; 0.817] | 

## Загрузка на Hugging Face Hub

Токен берётся из Colab Secrets (`HF_TOKEN`) и не выводится. Репозиторий создаётся, если его ещё нет.

In [10]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=EXPORT_DIR, repo_id=HF_REPO, repo_type="model",
                  commit_message=f"Классификатор интентов TuranAssist: {best}, ONNX int8")
print("готово: https://huggingface.co/" + HF_REPO)

готово: https://huggingface.co/AlexCode2003/turanassist-intent-e5
